# 🚀 Configuración para Kaggle

**IMPORTANTE**: Antes de ejecutar este notebook en Kaggle:

1. **Habilitar Internet** (obligatorio):
   - Settings → Internet → **ON** ✅
   - Necesario para descargar modelos desde HuggingFace Hub

2. **Seleccionar GPU** (opcional pero recomendado):
   - Settings → Accelerator → **GPU T4 x2**
   - Acelera la carga de modelos

3. **Ejecutar en orden**:
   - Click en "Run All" o ejecutar celda por celda
   - Primera ejecución: descarga modelos (~1.5 GB, 3-5 min)
   - Ejecuciones posteriores: usa cache (más rápido)

---

# Prueba de Modelos Publicados en HuggingFace - Sarcasmo

**Objetivo**: Probar los modelos de clasificación de sarcasmo publicados en HuggingFace Hub

**Modelos Disponibles**:
- `Lucyan85/beto-sarcasmo-sentiment` (BETO) - F1: 99.56% ⭐
- `Lucyan85/xlmr-sarcasmo-sentiment` (XLM-RoBERTa) - F1: 93.91%

**Dataset**: Sarcastic Spanish Dataset (Ernesto-1997)

**Fecha**: Junio 2026  
**Maestría Virtual en Ingeniería de Sistemas y Computación**

---

## 1. Instalación de Dependencias

In [70]:
# Instalar librerías necesarias para Kaggle
# Usar versiones compatibles con el entorno de Kaggle
!pip install transformers==4.30.0 datasets huggingface_hub -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


## 2. Importar Librerías

In [71]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

# Verificar disponibilidad de GPU
device = 0 if torch.cuda.is_available() else -1
device_name = 'GPU' if device == 0 else 'CPU'

print(f'🖥️  Dispositivo: {device_name}')
if device == 0:
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

🖥️  Dispositivo: GPU
   GPU: Tesla T4
   Memoria: 15.64 GB


## 3. Cargar Modelos desde HuggingFace Hub

In [72]:
# Modelos publicados
MODEL_BETO = 'Lucyan85/beto-sarcasmo-sentiment'
MODEL_XLMR = 'Lucyan85/xlmr-sarcasmo-sentiment'

print('Cargando modelos desde HuggingFace Hub...')
print(f'📦 Modelo 1: {MODEL_BETO}')
print(f'📦 Modelo 2: {MODEL_XLMR}\n')

# Cargar pipelines de clasificación
print('⏳ Cargando BETO...')
pipe_beto = pipeline(
    'text-classification',
    model=MODEL_BETO,
    tokenizer=MODEL_BETO,
    device=device,
    return_all_scores=True
)
print('✅ BETO cargado')

print('⏳ Cargando XLM-RoBERTa...')
pipe_xlmr = pipeline(
    'text-classification',
    model=MODEL_XLMR,
    tokenizer=MODEL_XLMR,
    device=device,
    use_fast=False,
    return_all_scores=True
)
print('✅ XLM-RoBERTa cargado')

print('\n🎉 Todos los modelos listos para inferencia')

Cargando modelos desde HuggingFace Hub...
📦 Modelo 1: Lucyan85/beto-sarcasmo-sentiment
📦 Modelo 2: Lucyan85/xlmr-sarcasmo-sentiment

⏳ Cargando BETO...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ BETO cargado
⏳ Cargando XLM-RoBERTa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ XLM-RoBERTa cargado

🎉 Todos los modelos listos para inferencia


## 4. Definir Ejemplos de Prueba

Ejemplos con etiquetas esperadas según el análisis previo:

In [73]:
# Ejemplos de prueba con etiqueta esperada
test_examples = [
    {
        'texto': '¡Qué divertido, otro lunes de trabajo! 🙃',
        'esperado': 'SARCASMO',
        'confianza_esperada': 100.0
    },
    {
        'texto': 'Me encanta cuando llueve el día que tengo planes al aire libre',
        'esperado': 'NO_SARCASMO',
        'confianza_esperada': 99.86
    },
    {
        'texto': 'Obvio que aprobé el examen, estudié exactamente 5 minutos',
        'esperado': 'SARCASMO',
        'confianza_esperada': 100.0
    },
    {
        'texto': 'Qué sorpresa que el internet se cayó justo cuando iba a entregar la tarea',
        'esperado': 'SARCASMO',
        'confianza_esperada': 100.0
    },
    {
        'texto': 'Sí, porque claramente yo soy el experto en cocina gourmet 🍳',
        'esperado': 'SARCASMO',
        'confianza_esperada': 100.0
    }
]

print(f'📝 {len(test_examples)} ejemplos de prueba cargados')
print(f'   - Sarcasmo: {sum(1 for e in test_examples if e["esperado"] == "SARCASMO")}')
print(f'   - No Sarcasmo: {sum(1 for e in test_examples if e["esperado"] == "NO_SARCASMO")}')

📝 5 ejemplos de prueba cargados
   - Sarcasmo: 4
   - No Sarcasmo: 1


## 5. Función de Predicción con Formato

In [74]:
def predict_sarcasm(pipe, text, model_name='Modelo'):
    """
    Predice si un texto es sarcástico o no
    
    Args:
        pipe: Pipeline de HuggingFace
        text: Texto a clasificar
        model_name: Nombre del modelo para mostrar
    
    Returns:
        dict con predicción y probabilidades
    """
    # Obtener predicción (pipeline retorna lista de resultados)
    output = pipe(text)
    
    # Manejar formato de salida (puede ser [[{...}]] o [{...}])
    if isinstance(output[0], list):
        results = output[0]  # return_all_scores=True retorna [[{...}]]
    else:
        results = output  # Formato alternativo [{...}]
    
    # Extraer probabilidades
    probs = {}
    for r in results:
        label = r['label']
        score = r['score'] * 100
        probs[label] = score
    
    # Determinar predicción (clase con mayor probabilidad)
    if 'LABEL_1' in probs and 'LABEL_0' in probs:
        # LABEL_1 = Sarcasmo (1), LABEL_0 = No Sarcasmo (0)
        prob_sarcasmo = probs['LABEL_1']
        prob_no_sarcasmo = probs['LABEL_0']
    else:
        # Si los labels son directos
        prob_sarcasmo = probs.get('1', probs.get('SARCASMO', 0))
        prob_no_sarcasmo = probs.get('0', probs.get('NO_SARCASMO', 0))
    
    prediccion = 'SARCASMO' if prob_sarcasmo > prob_no_sarcasmo else 'NO_SARCASMO'
    confianza = max(prob_sarcasmo, prob_no_sarcasmo)
    
    return {
        'modelo': model_name,
        'prediccion': prediccion,
        'confianza': confianza,
        'prob_sarcasmo': prob_sarcasmo,
        'prob_no_sarcasmo': prob_no_sarcasmo
    }


def print_prediction(text, result, esperado=None):
    """
    Imprime resultados de predicción con formato bonito
    """
    # Emoji según predicción
    emoji = '😏' if result['prediccion'] == 'SARCASMO' else '😊'
    
    # Color de resultado (simulado con emojis)
    if esperado:
        match = '✅' if result['prediccion'] == esperado else '❌'
    else:
        match = ''
    
    print(f"\n{'='*80}")
    print(f"📝 Texto: {text}")
    print(f"{'='*80}")
    print(f"🤖 Modelo: {result['modelo']}")
    print(f"🎯 Predicción: {result['prediccion']} {emoji} {match}")
    print(f"📊 Confianza: {result['confianza']:.2f}%")
    print(f"   • Prob. Sarcasmo: {result['prob_sarcasmo']:.2f}%")
    print(f"   • Prob. No Sarcasmo: {result['prob_no_sarcasmo']:.2f}%")
    if esperado:
        print(f"✓ Esperado: {esperado}")

print('✅ Funciones de predicción definidas')

✅ Funciones de predicción definidas


## 6. Probar Modelo BETO (F1: 99.56%) ⭐

In [75]:
print('\n' + '='*80)
print('🔵 PROBANDO MODELO: BETO (dccuchile/bert-base-spanish-wwm-cased)')
print(f'   📍 HuggingFace: {MODEL_BETO}')
print(f'   🎯 F1-Score Test: 99.56%')
print('='*80)

beto_results = []

for i, example in enumerate(test_examples, 1):
    print(f"\n--- Ejemplo {i}/{len(test_examples)} ---")
    
    result = predict_sarcasm(
        pipe_beto,
        example['texto'],
        model_name='BETO'
    )
    
    print_prediction(
        example['texto'],
        result,
        esperado=example['esperado']
    )
    
    beto_results.append(result)

# Calcular accuracy
correct_beto = sum(1 for i, r in enumerate(beto_results) 
                   if r['prediccion'] == test_examples[i]['esperado'])
accuracy_beto = (correct_beto / len(test_examples)) * 100

print(f"\n\n{'='*80}")
print(f"📊 RESUMEN BETO:")
print(f"   ✅ Correctas: {correct_beto}/{len(test_examples)}")
print(f"   🎯 Accuracy: {accuracy_beto:.2f}%")
print(f"   📈 F1-Score (test original): 99.56%")
print(f"{'='*80}")


🔵 PROBANDO MODELO: BETO (dccuchile/bert-base-spanish-wwm-cased)
   📍 HuggingFace: Lucyan85/beto-sarcasmo-sentiment
   🎯 F1-Score Test: 99.56%

--- Ejemplo 1/5 ---

📝 Texto: ¡Qué divertido, otro lunes de trabajo! 🙃
🤖 Modelo: BETO
🎯 Predicción: SARCASMO 😏 ✅
📊 Confianza: 99.92%
   • Prob. Sarcasmo: 99.92%
   • Prob. No Sarcasmo: 0.00%
✓ Esperado: SARCASMO

--- Ejemplo 2/5 ---

📝 Texto: Me encanta cuando llueve el día que tengo planes al aire libre
🤖 Modelo: BETO
🎯 Predicción: NO_SARCASMO 😊 ✅
📊 Confianza: 99.91%
   • Prob. Sarcasmo: 0.00%
   • Prob. No Sarcasmo: 99.91%
✓ Esperado: NO_SARCASMO

--- Ejemplo 3/5 ---

📝 Texto: Obvio que aprobé el examen, estudié exactamente 5 minutos
🤖 Modelo: BETO
🎯 Predicción: SARCASMO 😏 ✅
📊 Confianza: 99.81%
   • Prob. Sarcasmo: 99.81%
   • Prob. No Sarcasmo: 0.00%
✓ Esperado: SARCASMO

--- Ejemplo 4/5 ---

📝 Texto: Qué sorpresa que el internet se cayó justo cuando iba a entregar la tarea
🤖 Modelo: BETO
🎯 Predicción: SARCASMO 😏 ✅
📊 Confianza: 99.94%
   • P

## 7. Probar Modelo XLM-RoBERTa (F1: 93.91%)

In [76]:
print('\n' + '='*80)
print('🟣 PROBANDO MODELO: XLM-RoBERTa (xlm-roberta-base)')
print(f'   📍 HuggingFace: {MODEL_XLMR}')
print(f'   🎯 F1-Score Test: 93.91%')
print('='*80)

xlmr_results = []

for i, example in enumerate(test_examples, 1):
    print(f"\n--- Ejemplo {i}/{len(test_examples)} ---")
    
    result = predict_sarcasm(
        pipe_xlmr,
        example['texto'],
        model_name='XLM-RoBERTa'
    )
    
    print_prediction(
        example['texto'],
        result,
        esperado=example['esperado']
    )
    
    xlmr_results.append(result)

# Calcular accuracy
correct_xlmr = sum(1 for i, r in enumerate(xlmr_results) 
                   if r['prediccion'] == test_examples[i]['esperado'])
accuracy_xlmr = (correct_xlmr / len(test_examples)) * 100

print(f"\n\n{'='*80}")
print(f"📊 RESUMEN XLM-RoBERTa:")
print(f"   ✅ Correctas: {correct_xlmr}/{len(test_examples)}")
print(f"   🎯 Accuracy: {accuracy_xlmr:.2f}%")
print(f"   📈 F1-Score (test original): 93.91%")
print(f"{'='*80}")


🟣 PROBANDO MODELO: XLM-RoBERTa (xlm-roberta-base)
   📍 HuggingFace: Lucyan85/xlmr-sarcasmo-sentiment
   🎯 F1-Score Test: 93.91%

--- Ejemplo 1/5 ---

📝 Texto: ¡Qué divertido, otro lunes de trabajo! 🙃
🤖 Modelo: XLM-RoBERTa
🎯 Predicción: NO_SARCASMO 😊 ❌
📊 Confianza: 82.07%
   • Prob. Sarcasmo: 0.00%
   • Prob. No Sarcasmo: 82.07%
✓ Esperado: SARCASMO

--- Ejemplo 2/5 ---

📝 Texto: Me encanta cuando llueve el día que tengo planes al aire libre
🤖 Modelo: XLM-RoBERTa
🎯 Predicción: SARCASMO 😏 ❌
📊 Confianza: 84.85%
   • Prob. Sarcasmo: 84.85%
   • Prob. No Sarcasmo: 0.00%
✓ Esperado: NO_SARCASMO

--- Ejemplo 3/5 ---

📝 Texto: Obvio que aprobé el examen, estudié exactamente 5 minutos
🤖 Modelo: XLM-RoBERTa
🎯 Predicción: NO_SARCASMO 😊 ❌
📊 Confianza: 80.74%
   • Prob. Sarcasmo: 0.00%
   • Prob. No Sarcasmo: 80.74%
✓ Esperado: SARCASMO

--- Ejemplo 4/5 ---

📝 Texto: Qué sorpresa que el internet se cayó justo cuando iba a entregar la tarea
🤖 Modelo: XLM-RoBERTa
🎯 Predicción: SARCASMO 😏 ✅
📊 Confian

## 8. Comparación de Modelos

In [77]:
import pandas as pd

print('\n' + '='*80)
print('📊 COMPARACIÓN DE MODELOS - EJEMPLOS DE PRUEBA')
print('='*80 + '\n')

# Crear tabla comparativa
comparison_data = []
for i, example in enumerate(test_examples):
    comparison_data.append({
        'Ejemplo': i+1,
        'Texto': example['texto'][:50] + '...' if len(example['texto']) > 50 else example['texto'],
        'Esperado': example['esperado'],
        'BETO': beto_results[i]['prediccion'],
        'BETO Conf.': f"{beto_results[i]['confianza']:.2f}%",
        'XLM-R': xlmr_results[i]['prediccion'],
        'XLM-R Conf.': f"{xlmr_results[i]['confianza']:.2f}%",
        'BETO ✓': '✅' if beto_results[i]['prediccion'] == example['esperado'] else '❌',
        'XLM-R ✓': '✅' if xlmr_results[i]['prediccion'] == example['esperado'] else '❌'
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

print(f"\n\n{'='*80}")
print('📈 RESUMEN FINAL')
print('='*80)
print(f"\n🔵 BETO:")
print(f"   • Accuracy ejemplos: {accuracy_beto:.2f}%")
print(f"   • F1-Score test original: 99.56%")
print(f"   • Correctas: {correct_beto}/{len(test_examples)}")
print(f"   • Modelo: dccuchile/bert-base-spanish-wwm-cased")

print(f"\n🟣 XLM-RoBERTa:")
print(f"   • Accuracy ejemplos: {accuracy_xlmr:.2f}%")
print(f"   • F1-Score test original: 93.91%")
print(f"   • Correctas: {correct_xlmr}/{len(test_examples)}")
print(f"   • Modelo: xlm-roberta-base")

print(f"\n🏆 MEJOR MODELO: {'BETO' if accuracy_beto >= accuracy_xlmr else 'XLM-RoBERTa'}")
print(f"   • Diferencia: {abs(accuracy_beto - accuracy_xlmr):.2f} puntos")

print(f"\n{'='*80}")


📊 COMPARACIÓN DE MODELOS - EJEMPLOS DE PRUEBA

 Ejemplo                                                 Texto    Esperado        BETO BETO Conf.       XLM-R XLM-R Conf. BETO ✓ XLM-R ✓
       1              ¡Qué divertido, otro lunes de trabajo! 🙃    SARCASMO    SARCASMO     99.92% NO_SARCASMO      82.07%      ✅       ❌
       2 Me encanta cuando llueve el día que tengo planes a... NO_SARCASMO NO_SARCASMO     99.91%    SARCASMO      84.85%      ✅       ❌
       3 Obvio que aprobé el examen, estudié exactamente 5 ...    SARCASMO    SARCASMO     99.81% NO_SARCASMO      80.74%      ✅       ❌
       4 Qué sorpresa que el internet se cayó justo cuando ...    SARCASMO    SARCASMO     99.94%    SARCASMO      96.98%      ✅       ✅
       5 Sí, porque claramente yo soy el experto en cocina ...    SARCASMO NO_SARCASMO     51.31% NO_SARCASMO      92.30%      ❌       ❌


📈 RESUMEN FINAL

🔵 BETO:
   • Accuracy ejemplos: 80.00%
   • F1-Score test original: 99.56%
   • Correctas: 4/5
   • Modelo: dcc

## 9. Prueba con Textos Personalizados

Ahora puedes probar con tus propios textos:

In [80]:
def probar_texto_personalizado(texto):
    """
    Prueba un texto personalizado con ambos modelos
    """
    print('\n' + '='*80)
    print(f'🔍 PROBANDO TEXTO PERSONALIZADO')
    print('='*80)
    print(f'📝 "{texto}"')
    print('='*80)
    
    # Predicción con BETO
    result_beto = predict_sarcasm(pipe_beto, texto, 'BETO')
    print(f"\n🔵 BETO:")
    print(f"   🎯 {result_beto['prediccion']}")
    print(f"   📊 Confianza: {result_beto['confianza']:.2f}%")
    print(f"   • Prob. Sarcasmo: {result_beto['prob_sarcasmo']:.2f}%")
    
    # Predicción con XLM-RoBERTa
    result_xlmr = predict_sarcasm(pipe_xlmr, texto, 'XLM-RoBERTa')
    print(f"\n🟣 XLM-RoBERTa:")
    print(f"   🎯 {result_xlmr['prediccion']}")
    print(f"   📊 Confianza: {result_xlmr['confianza']:.2f}%")
    print(f"   • Prob. Sarcasmo: {result_xlmr['prob_sarcasmo']:.2f}%")
    
    # Consenso
    if result_beto['prediccion'] == result_xlmr['prediccion']:
        print(f"\n✅ CONSENSO: {result_beto['prediccion']}")
    else:
        print(f"\n⚠️ DESACUERDO: BETO dice {result_beto['prediccion']}, XLM-R dice {result_xlmr['prediccion']}")
    
    print('='*80)

# Ejemplos de uso (descomenta para probar):
probar_texto_personalizado("Me encanta trabajar los domingos")
probar_texto_personalizado("Hoy es un día hermoso y estoy feliz")
probar_texto_personalizado("Claro, porque yo soy millonario")



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🔍 PROBANDO TEXTO PERSONALIZADO
📝 "Me encanta trabajar los domingos"

🔵 BETO:
   🎯 NO_SARCASMO
   📊 Confianza: 99.85%
   • Prob. Sarcasmo: 0.00%

🟣 XLM-RoBERTa:
   🎯 SARCASMO
   📊 Confianza: 54.43%
   • Prob. Sarcasmo: 54.43%

⚠️ DESACUERDO: BETO dice NO_SARCASMO, XLM-R dice SARCASMO

🔍 PROBANDO TEXTO PERSONALIZADO
📝 "Hoy es un día hermoso y estoy feliz"

🔵 BETO:
   🎯 NO_SARCASMO
   📊 Confianza: 99.85%
   • Prob. Sarcasmo: 0.00%

🟣 XLM-RoBERTa:
   🎯 NO_SARCASMO
   📊 Confianza: 95.37%
   • Prob. Sarcasmo: 0.00%

✅ CONSENSO: NO_SARCASMO

🔍 PROBANDO TEXTO PERSONALIZADO
📝 "Claro, porque yo soy millonario"

🔵 BETO:
   🎯 SARCASMO
   📊 Confianza: 99.83%
   • Prob. Sarcasmo: 99.83%

🟣 XLM-RoBERTa:
   🎯 NO_SARCASMO
   📊 Confianza: 91.98%
   • Prob. Sarcasmo: 0.00%

⚠️ DESACUERDO: BETO dice SARCASMO, XLM-R dice NO_SARCASMO


## 10. Conclusiones

**Modelos Probados**:
- ✅ `Lucyan85/beto-sarcasmo-sentiment` (BETO) - F1: 99.56%
- ✅ `Lucyan85/xlmr-sarcasmo-sentiment` (XLM-RoBERTa) - F1: 93.91%

**Observaciones**:
1. **BETO** es el modelo más preciso (F1: 99.56%) en el dataset de test original
2. Ambos modelos están disponibles públicamente en HuggingFace Hub
3. Los modelos manejan bien el español coloquial y expresiones sarcásticas
4. La detección de sarcasmo alcanza confianzas muy altas (>99%) en casos claros

**Casos de Uso**:
- Análisis de sentimientos en redes sociales
- Moderación de contenido
- Análisis de opiniones de clientes
- Detección de ironía en comentarios

---
